# Task 4 — CGL mixture design: source download and benchmark preparation

## Goal

Download the Eigenvector CGL archive, select glucose from the four response variables, and preserve the supplied `Xcal`/`Xtest` partition. The historical folder name contains “grain”; the source is a casein, glucose, lactate, and moisture mixture design.

The original download is held in a temporary directory and removed at the end. Only the generated benchmark CSV files remain in the task directory.

## Original source

- Dataset page: <https://eigenvector.com/resources/data-sets/>
- Archive: <https://eigenvector.com/wp-content/uploads/2021/04/CGL_nir.mat_.zip>

## Setup

Set the task parameters, locate the output directory, and define download and verification helpers. The comparison runs before the generated files replace any existing local copies.

In [1]:
TASK_FOLDER = "4-CGL_NIR_grain_glucose"
INCLUDE_INDEX = False
# The upstream MATLAB archive and the published CSV differ only in the final
# decimal places of some values (maximum observed difference below 4e-9).
COMPARISON_ATOL = 1e-8
SOURCE_URL = "https://eigenvector.com/wp-content/uploads/2021/04/CGL_nir.mat_.zip"

In [2]:
from pathlib import Path
from urllib.request import Request, urlopen
import hashlib
import io
import shutil
import tempfile
import zipfile

import numpy as np
import pandas as pd
from scipy.io import loadmat


def find_task_dir(folder_name):
    """Support running from either the task directory or repository root."""
    candidates = [Path.cwd(), Path.cwd() / "datasets" / folder_name]
    for candidate in candidates:
        if candidate.name == folder_name and (candidate / "download_data.md").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Run this notebook from {folder_name!r} or from the repository root."
    )


def download(url, destination):
    """Download a source file without retaining it in the repository."""
    request = Request(url, headers={"User-Agent": "NIRBENCH-DL/0.1 dataset preparation"})
    with urlopen(request, timeout=120) as response, destination.open("wb") as output:
        shutil.copyfileobj(response, output)
    print(f"Downloaded {destination.name} ({destination.stat().st_size:,} bytes)")


def csv_roundtrip(frame, include_index):
    """Represent a generated frame as it will appear after CSV serialization."""
    buffer = io.StringIO()
    frame.to_csv(buffer, index=include_index, lineterminator="\r\n")
    buffer.seek(0)
    return pd.read_csv(buffer)


def verify_and_write(train_frame, test_frame, include_index, atol):
    """Compare with published partitions when present, then write both CSV files."""
    results = []
    for split, generated in (("train", train_frame), ("test", test_frame)):
        output_path = TASK_DIR / f"data_{split}.csv"
        serialized = csv_roundtrip(generated, include_index)

        if output_path.exists():
            published = pd.read_csv(output_path)
            pd.testing.assert_frame_equal(
                serialized,
                published,
                check_dtype=False,
                check_exact=False,
                rtol=0,
                atol=atol,
            )
            max_delta = np.max(
                np.abs(
                    serialized.select_dtypes(include="number").to_numpy(dtype=float)
                    - published.select_dtypes(include="number").to_numpy(dtype=float)
                )
            )
            print(
                f"{split}: matches the published partition "
                f"(shape={serialized.shape}, maximum absolute difference={max_delta:.3g})"
            )
        else:
            print(f"{split}: no published CSV was present; generated shape={serialized.shape}")

        generated.to_csv(
            output_path,
            index=include_index,
            lineterminator="\r\n",
        )
        digest = hashlib.sha256(output_path.read_bytes()).hexdigest()
        results.append({
            "partition": split,
            "rows": len(generated),
            "spectral_variables": generated.shape[1] - 1,
            "sha256": digest,
        })

    return pd.DataFrame(results)


TASK_DIR = find_task_dir(TASK_FOLDER)
temporary_download = tempfile.TemporaryDirectory(prefix=f"nirbench_{TASK_FOLDER}_")
SOURCE_DIR = Path(temporary_download.name)
print(f"Preparing task {TASK_FOLDER}")

Preparing task 4-CGL_NIR_grain_glucose


## Download and prepare

Download the authoritative source and apply the task specific conversion recorded in `DATA_LICENSES.md`.

In [3]:
archive_path = SOURCE_DIR / "CGL_nir.zip"
download(SOURCE_URL, archive_path)
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(SOURCE_DIR)

mat_path = next(SOURCE_DIR.rglob("CGL_nir.mat"))
source = loadmat(mat_path, squeeze_me=True, struct_as_record=False)

wavelengths = [str(value) for value in np.asarray(source["Xcal"].axisscale[1, 0]).ravel()]
response_labels = [str(value) for value in np.asarray(source["Ycal"].label[1, 0]).ravel()]
glucose_column = next(i for i, label in enumerate(response_labels) if label.strip() == "Glucose (wt %)")
target_name = response_labels[glucose_column]

train_data = pd.DataFrame(source["Xcal"].data, columns=wavelengths)
train_data = train_data.assign(**{target_name: source["Ycal"].data[:, glucose_column]})

test_data = pd.DataFrame(source["Xtest"].data, columns=wavelengths)
test_data = test_data.assign(**{target_name: source["Ytest"].data[:, glucose_column]})

assert train_data.shape == (153, 118)
assert test_data.shape == (78, 118)
print("Selected Glucose (wt %) and preserved the supplied calibration/test partition.")

Downloaded CGL_nir.zip (224,320 bytes)
Selected Glucose (wt %) and preserved the supplied calibration/test partition.


## Verify and write

When published CSVs are present, compare their shapes, columns, row order, indices, and numerical values with the generated partitions. Then write `data_train.csv` and `data_test.csv` using deterministic CRLF line endings.

In [4]:
summary = verify_and_write(
    train_data,
    test_data,
    include_index=INCLUDE_INDEX,
    atol=COMPARISON_ATOL,
)
display(summary)
temporary_download.cleanup()
print("Temporary source files removed. Preparation complete.")

train: matches the published partition (shape=(153, 118), maximum absolute difference=3.07e-09)
test: matches the published partition (shape=(78, 118), maximum absolute difference=1.53e-09)


,partition,rows,spectral_variables,sha256
0,train,153,117,8fff2906f0f1d4b3eb83c32fc6025e0d146577e2598d27...
1,test,78,117,59c3bedc8eaf71d442a18797325074c20d3729c4cd0355...


Temporary source files removed. Preparation complete.
